In [ ]:

import requests
from bs4 import BeautifulSoup
import csv
import time
import re

BASE_URL  = "https://nanoreview.net"
START_URL = f"{BASE_URL}/en/soc-list/rating"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://nanoreview.net/",
}


def fetch(url: str, retries: int = 3, delay: float = 2.5) -> str | None:
    """GET a page with retry / back-off."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            print(f"  ✓ {url}  [{r.status_code}]")
            return r.text
        except requests.RequestException as exc:
            print(f"  ✗ Attempt {attempt}/{retries} failed: {exc}")
            if attempt < retries:
                time.sleep(delay * attempt)
    return None


def clean(text: str) -> str:
    return " ".join(text.split())


# ── Parser ────────────────────────────────────────────────────────────────────

def parse_soc_table(soup: BeautifulSoup) -> list[dict]:

    rows = []
    table = soup.select_one("table.table-list tbody")
    if not table:
        print("  [!] Table body not found – check selector")
        return rows

    for tr in table.find_all("tr"):
        cols = tr.find_all("td")
        if len(cols) < 8:
            continue

        rank_label = cols[0].select_one("label")
        rank = clean(rank_label.get_text()) if rank_label else clean(cols[0].get_text())

        checkbox = cols[0].find("input", type="checkbox")
        full_name = checkbox["aria-label"] if checkbox and checkbox.get("aria-label") else ""

        name_a = cols[1].find("a")
        name    = clean(name_a.get_text()) if name_a else clean(cols[1].get_text())
        soc_url = BASE_URL + name_a["href"] if name_a and name_a.get("href") else ""

        brand_span = cols[1].find("span", class_="text-gray-small")
        brand = clean(brand_span.get_text()) if brand_span else ""

        rating_score = cols[2].get("data-sort", "")
        score_box    = cols[2].find("div", class_="table-list-score-box")
        rating_num   = clean(score_box.get_text()) if score_box else rating_score
        grade_span   = cols[2].find("span")
        grade        = clean(grade_span.get_text()) if grade_span else ""

        antutu_raw = cols[3].get("data-sort", "")
        for bar in cols[3].find_all("div", class_="score-bar-line"):
            bar.decompose()
        antutu_display = clean(cols[3].get_text())
        antutu = antutu_raw or antutu_display

        gb_raw = cols[4].get("data-sort", "")       
        for bar in cols[4].find_all("div", class_="score-bar-line"):
            bar.decompose()
        gb_text = clean(cols[4].get_text())      
        gb_single = gb_multi = ""
        if "/" in gb_text:
            parts = [p.strip() for p in gb_text.split("/")]
            gb_single = parts[0]
            gb_multi  = parts[1] if len(parts) > 1 else ""

        core_count  = cols[5].get("data-sort", "")
        circle_div  = cols[5].find("div", class_="table-list-custom-circle")
        core_config_raw = clean(cols[5].get_text())
        if circle_div:
            core_config = core_config_raw.replace(clean(circle_div.get_text()), "").strip()
        else:
            core_config = core_config_raw

        clock = clean(cols[6].get_text())

        gpu = clean(cols[7].get_text())

        rows.append({
            "rank":         rank,
            "Processor_name":         name,
            "Processor_full_name":    full_name, 
            "brand":        brand,
            "url":          soc_url,
            "rating":       rating_num,
            "grade":        grade,
            "antutu_v11":   antutu,
            "gb6_single":   gb_single,
            "gb6_multi":    gb_multi,
            "cores":        core_count,
            "core_config":  core_config,
            "clock_mhz":    clock,
            "gpu":          gpu,
        })

    return rows


def get_total_pages(soup: BeautifulSoup) -> int:
    pagination = soup.select_one(".pagination-map")
    if pagination:
        text = pagination.get_text()
        m = re.search(r"of\s+(\d+)", text)
        if m:
            return int(m.group(1))
    return 1


def scrape(max_pages: int | None = None) -> list[dict]:
    print(f"\nFetching page 1 …")
    html = fetch(START_URL)
    if not html:
        print("Failed to fetch the first page.")
        return []

    soup        = BeautifulSoup(html, "html.parser")
    total_pages = get_total_pages(soup)
    if max_pages:
        total_pages = min(total_pages, max_pages)

    print(f"Total pages detected: {total_pages}")
    all_data = parse_soc_table(soup)
    print(f"  → {len(all_data)} records from page 1")

    for page in range(2, total_pages + 1):
        url = f"{START_URL}?page={page}"
        print(f"\nFetching page {page} …")
        html = fetch(url)
        if not html:
            print(f"  Skipping page {page}")
            continue
        page_soup = BeautifulSoup(html, "html.parser")
        rows = parse_soc_table(page_soup)
        print(f"  → {len(rows)} records")
        all_data.extend(rows)
        time.sleep(1.5)          # polite delay

    return all_data


def save(data: list[dict], csv_path: str) -> None:
    if not data:
        print("No data to save.")
        return

    fieldnames = [
        "rank", "Processor_name", "Processor_full_name", "brand", "url",
        "rating", "grade", "antutu_v11",
        "gb6_single", "gb6_multi",
        "cores", "core_config", "clock_mhz", "gpu",
    ]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f"\n✓ CSV saved → {csv_path}  ({len(data)} rows)")


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Set max_pages=None to scrape everything, or e.g. max_pages=2 to test
    data = scrape(max_pages=None)

    if data:
        print(f"\n{'─'*60}")
        print(f"Total SoCs scraped: {len(data)}")
        print(f"\nTop 5 preview:")
        print(f"{'#':<4} {'Chip':<35} {'Brand':<12} {'Rating':<8} {'Grade':<6} {'AnTuTu':>10}")
        print("─" * 85)
        for d in data[:5]:
            print(f"{d['rank']:<4} {d['Processor_name']:<35} {d['brand']:<12} "
                  f"{d['rating']:<8} {d['grade']:<6} {d['antutu_v11']:>10}")

        save(
            data,
            csv_path = "../GSMArenaDataset/Rate_on_antutu_score.csv",
        )
    else:
        print("No data scraped.")


Fetching page 1 …
  ✓ https://nanoreview.net/en/soc-list/rating  [200]
Total pages detected: 2
  → 200 records from page 1

Fetching page 2 …
  ✓ https://nanoreview.net/en/soc-list/rating?page=2  [200]
  → 33 records

────────────────────────────────────────────────────────────
Total SoCs scraped: 233

Top 5 preview:
#    Chip                                Brand        Rating   Grade      AnTuTu
─────────────────────────────────────────────────────────────────────────────────────
1    Snapdragon 8 Elite Gen 5            Qualcomm     97       A+        3932243
2    A19 Pro                             Apple        96       A+        2606807
3    Dimensity 9500                      MediaTek     96       A+        3508681
4    Exynos 2600                         Samsung      93       A+        3145925
5    Snapdragon 8 Elite (Gen 4)          Qualcomm     93       A+        3336910

✓ CSV saved → ../GSMArenaDataset/Rate_on_antutu_score.csv  (233 rows)
